In [1]:
import numpy as np
rng = np.random.default_rng(2024)

np.version.full_version

'2.5.3'

### ndarray
1. An ndarray is a generic multidimensional container for **homogeneous** data; that is, all of the elements must be the same type. Every array has a shape, a **tuple** indicating the size of each dimension, and a `dtype`, an object describing the data type of the array.
1. An ndarray is four things: 
    - a contiguous block of memory, 
    - a dtype (how to interpret each fixed-width chunk of bytes), 
    - a shape, 
    - strides (how many bytes to jump to move one step along each axis). 
1. Speed comes from handing that block to a **compiled C loop with no per-element type checking**. Every "gotcha" in this chapter — views, dtype surprises, axis semantics — is a direct consequence of this design. 
1. A Python list of floats is, by contrast, an array of pointers to separately allocated objects.

In [ ]:
%timeit np.arange(1_000_000)*2 
%timeit [x * 2 for x in range(1_000_000)]
# The speed difference comes from  
# 1. Homogeneous dtype → no per-element type dispatch or boxing; 
# 2. the loop runs in compiled C over contiguous, cache-friendly memory

364 μs ± 23.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
38.8 ms ± 363 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [34]:
a1 = np.arange(1e6) # nbytes = 8M
a2 = np.arange(1e6, dtype='int32') # nbytes = 4M
a1.nbytes, a2.nbytes

(8000000, 4000000)


### Creating Array
1. `np.array(seq)` infers `dtype` (upcasting to the most general type present) and infers `ndim` from nesting depth. It by default ***copies*** data!
    - `a2` below converts everything into a Unicode string (up to length 21) because promotion to string is safe here.
    - The conversion can be made explicitly see `a4` and `a5`.
1. `np.asarray` returns the input itself if it's already an ndarray (**no copy**); np.array copies.
1. Factories: `arange`, `zeros`, `ones`, `empty` (uninitialized garbage — not zeros), `full`, and their `_like` variants (copy shape and dtype of a template), `eye`/`identity`.
    - NOTE: `_like` functions will prioritize preserving the template's `dtype` so when it can't safely cast values it will raise a warning e.g. from `int` to `nan`.
1. Shapes are tuples: `np.zeros((3, 4))`. 
    - It infers shape from the outer to inner nesting structure.
    - `[[[1, 2, 3]],[[4, 5, 6]]]` would have a shape of `(2, 1, 3)` because the outer list has two elements; each of which has one element and each of which has three elements.
1. Inspect with .shape, .ndim, .dtype, .size, .nbytes

In [31]:
a1 = np.array([[1, 2], [3, 4.0]]) # dtype = float64
a2 = np.array([1, 2, "3"]) # dtype = object --- WRONG! it's actually an array of string e.g. U21.
a3 = np.array([True, 2]) # dtype = int64 
for a in [a1, a2, a3]:
    print(f'Array={a}, Shape={a.shape}, ndim={a.ndim}, dtype={a.dtype}, size={a.size}, nbytes={a.nbytes}')


Array=[[1. 2.]
 [3. 4.]], Shape=(2, 2), ndim=2, dtype=float64, size=4, nbytes=32
Array=['1' '2' '3'], Shape=(3,), ndim=1, dtype=<U21, size=3, nbytes=252
Array=[1 2], Shape=(2,), ndim=1, dtype=int64, size=2, nbytes=16


In [28]:
a4 = np.array([1, 2, '3'], dtype=np.int64)
a5 = np.array([1, 2, '3'], dtype=object)

a4.dtype, a5.dtype

(dtype('int64'), dtype('O'))

In [6]:
a6 = np.array([[[1, 2, 3]], [[4, 5, 6]]])
assert a6.ndim == 3
#assert a6.shape == (2, 3, 2)
a6.shape

(2, 1, 3)

In [28]:
a = np.arange(5)
assert a[0] == 0
b = np.asarray(a) # doesn't copy!
c = np.array(a) # copy
assert c[0] == 0
b[0] = 99

assert a[0] == 99
assert c[0] == 0

# Build a 5×3 float array of prices all equal to 100.0 with no loop. 
a = np.full((5, 3), 100.0)

# Then make a same-shape template filled with NaN using a _like function. 
b = np.full_like(a, np.nan)

# Then run np.full_like(np.arange(3), np.nan) — what happens and why?
np.full_like(np.arange(3), np.nan)
np.full_like(np.arange(3, dtype=float), np.nan)

/Users/tiantang/.pyenv/versions/fluent-py-3.14/lib/python3.14/site-packages/numpy/_core/numeric.py:475: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


array([nan, nan, nan])

### Data types for ndarrays
1. Mental model: The `dtype` is a contract about bytes: `int8/16/32/64`, `uint*`, `float16/32/64`, `complex`, `bool`, `object`, fixed-width bytes `S`, fixed-width unicode `U`. 
1. `astype` always returns a **new** array, even for the same dtype. Float→int truncates toward zero. Numeric strings convert via `astype(float)`. Integer arithmetic silently wraps on overflow; float 1/0 gives `inf` with a warning, not an exception.

1. NumPy 2 notes: the book's Table 4-2 lists np.string_/np.unicode_ — those are gone; use `np.bytes_`/`np.str_`. Type promotion changed (NEP 50): Python scalars are "weak", so float32_array + 1.5 stays float32, and a Python int that doesn't fit the array dtype now raises `OverflowError` instead of silently upcasting.

In [36]:
# 2a. Predict 
assert np.all(np.array([-1.7, 1.7, 2.5, -0.2]).astype(np.int64) == (np.array([-1, 1, 2, 0])))

#2b. Predict 
assert np.all(np.array(["3.5", "4", "1e3", "-0.25"]).astype(float) == np.array([3.5, 4.0, 1000.0, -0.25]))

#2c. 
x = np.arange(3, dtype=np.uint8)
# Predict 

assert np.all(x - 5 == np.array([251, 252, 253]))
# Then run 
try:
    x - 300 
except OverflowError as e:
    print(e) #— behaviour differs between NumPy 1.x and 2.x; explain both.

#2d. Why does np.array([1, 2, 3]) / 2 give float64?
# Because it does true float division
# but np.array([1, 2, 3]) // 2 give int64?
# Because it does floor division element-wise

#2e. You want to hold a 2,500-day × 3,000-stock returns matrix in RAM. 
# Compute the MB as float64 and as float32.
b = np.empty((2500, 3000), dtype=np.float64)  # 60M
a = np.empty((2500, 3000), dtype=np.float32)  # 30M
print(a.nbytes, b.nbytes)
# Roughly how many significant digits does float32 carry, and when would that bite you?
# effectively 24 bits of precision because of the implicit leading bit and roughly 7 sigificant digits
# max float32 is about 16.77m
# float32 ≈ 7 significant digits — fine for storing returns, risky when accumulating sums/products over thousands of terms
print(np.float32(17_000_000) + np.float32(1))

#2f.
a = np.arange(3)
b = a.astype(a.dtype); # a new array i.e. deep copying values
b[0] = 100 
assert a[0] == 0

Python integer 300 out of bounds for uint8
30000000 60000000
1.7e+07


### Arithmetic with NumPy arrays (plus the broadcasting rule you need now)

1. Mental model: Arithmetic is **elementwise** and returns a **new** array; comparisons return boolean arrays. 

1. To understand broadcasting, one needs to understand the difference between `(k,)`, `(1, k)` and `(k, 1)` as they're three genuinely ***different objects***, not three spellings of the same thing.

    | shape | ndim | meaning |
    |---|---|---|
    | `(k,)` | 1 | a bare list of k numbers — neither a row nor column vector, it has **no row/column orientation** which only exists when you have axes! |
    | `(1, k)` | 2 | one row, k columns — orientation exists |
    | `(k, 1)` | 2 | k rows, one column — orientation exists |

1. So why does `(k,1)` act like (1, k) in broadcasting?

    - Because the broadcasting algorithm's first step is literally: "if the two arrays have different ndim, pad the shape of the smaller one with 1s on the **left** until the ranks match." 
    - That padding is a temporary, virtual reinterpretation done only for the purpose of comparing shapes — it doesn't change the actual array, its ndim, or its strides. `(k,)` momentarily gets treated as if it were `(1, k)` only when it's standing next to a higher-rank array. By itself, on its own, it is still genuinely 1-D with no orientation.

1. Broadcasting (for now): 
    - align shapes from the ***right***; 
    - two dimensions are compatible if they are equal or one of them is 1; 
    - missing leading dimensions count as 1. 
    - A scalar has shape () and broadcasts with anything.
    - A 1-D array of length k, when entering a 2-D calculation, always aligns to the **last** axis. In a 2-D matrix, the last axis is columns. So shape `(k,)` always means `(1, k)` e.g. "one value per column, apply it to every row" — never "one value per row" — unless you reshape it. 
        - This is why (5, 3) with (3,) works because (3,) is turned into (1, 3); (5, 3) with (5,) fails (5, 3) is incompatible with (1, 5) — reshape to (5, 1) via `x[:, np.newaxis]`, `x[:, None]`, or `keepdims=True` (meaning keeps the other axis alive rather than deleting it!).
    - Lesson: Broadcasting checks shapes, not meaning. For a square matrix of `R`, both `R.mean(axis=0)` and `R.mean(axis=1)` would work with `R`, So test broadcasting-heavy code with **non-square** shapes on purpose.

In [56]:
a0 = np.array([3, 5, 6])
print(a0)
print(a0.shape, a0.ndim)
a1 = a0.reshape(-1, 3)
print(a1)
a1.shape, a1.ndim

[3 5 6]
(3,) 1
[[3 5 6]]


((1, 3), 2)

In [76]:
# 3a. P is a 4-day × 3-asset price matrix. Write the simple-return matrix in one expression using slices, and predict its shape.
P = np.full((4, 3), rng.normal())
ret = P[1:] / P[:-1] - 1 # same as P[1:, :] / P[:-1, :] but the col index can be omitted!
assert ret.shape == (3, 3)


# 3b. 
R = np.array([[1, 2, 3], [4, 5, 6]]) # (2, 3)

print(R.mean(axis=0).shape) # (3,) because it collapses on rows so 1 mean per column
print(R.mean(axis=1).shape) # (2,) it collapses on columns so 1 mean per row

# For each line, predict works/fails and say what it means financially when it works:
row_mean = R.mean(axis=0) # row_mean = [2.5, 3.5, 4.5]
R - row_mean # works because R is 2x3 and row_mean is (3,) thus compatible with broadcasting rule when (3,) becomes (1, 3).
# [[-1.5, -1.5, -1.5], [1.5, 1.5, 1.5]]
try:
    col_mean = R.mean(axis=1) # collapses on column so 1 mena per row [2., 5.] this is (2,)
    R - col_mean # fails because R is 2x3 and col mean is (2,) which becomes (1, 2) and incompatible with (2, 3)
except ValueError as e:
    print(e)

col_mean2 = R.mean(axis=1, keepdims=True) # this is (2, 1) keepsdim=True makes row axis alive e.g. col_mean2 = [[2.0], [5.0]]
R - col_mean2 # works now as (2, 3) and (2, 1) are compatible e.g. [[-1.0, 0., 1], [-1., 0., 1.]] 

 #3c.Trap: repeat 3b with `R` of shape `(3, 3)`. Which line now runs *without error* but computes the wrong thing? 
 # Why is this dangerous in research code?
R2 = np.array([[1, 2, 0], [1, 2, 3], [4, 5, 6]])
row_mean = R2.mean(axis=0) # [2, 3, 3] and shape is (3,)
assert np.array_equal(row_mean, np.array([2, 3, 3]))
diff1 = R2 - row_mean # works because (3, 3) and (3,) are compatible;
assert np.array_equal(diff1, np.array([[-1., -1., -3.], [-1., -1., 0.], [2., 2., 3.]]))

col_mean = R2.mean(axis=1) # [1, 2, 5] and shape is (3,)
assert np.array_equal(col_mean, np.array([1, 2, 5.]))
diff2 = R2 - col_mean # also works
assert np.array_equal(diff2, np.array([[0, 0, -5], [0, 0, -2], [3, 3, 1]]))


 #3d.
# Predict `(a * b).shape` and the values.
a = np.array([1, 2, 3])
b = np.array([[10], [20]])
# a is (3,) thus (1, 3) and b is (2, 1) so a*b are compatible and result shape is (2,3)
res = np.array([[10, 20, 30], [20, 40, 60]])
assert np.array_equal(res, a*b)
 
#3e. Predict `np.array([0.1, 0.2]).sum() == 0.3`. Then write the correct way to make this comparison
assert np.isclose(np.array([0.1, 0.2]).sum(), 0.3)

(3,)
(2,)
operands could not be broadcast together with shapes (2,3) (2,) 


### Basic indexing and slicing

**Mental model.**
1. Slices are ***views***, so is **basic array indexing**: no data is copied, and writing through the view writes to the original — unlike Python list slices, which copy data!
    - Numpy is designed to work with *very large* arrays so if slicing copies it would have performance and memory issues.
1. Assigning a scalar to a slice broadcasts into every selected position.
1. Shape rule:
    - **an integer index removes an axis; a slice keeps it.**
    - `P[2]` is 1-D because the row axis is removed; `P[2:3]` is 2-D because the row axis is kept.
    - `P[:, 2]` is 1-D because the column axis is removed; `P[:, 2:3]` is 2-D because the column axis is kept.
    - `P[-1, ::2]` is 1-D because the first index `-1` removes the row axis, leaving only the column axis to slice. This is why it produces every second element in the last row, not a 2-D block.
    - `P[0][2]` and `P[0, 2]` give the same element, but only the comma form generalizes: `P[:, 2]` is column 2, while `P[:][2]` is row 2.
1. Rule of thumb: if you want a row/column object to remain 2-D, keep the axis with a slice; if you want a single element or a 1-D selection, use an integer index.
1. Use `.copy()` when you truly need independent data.

**Quick reference**
- `P[row]` -> 1-D row selection
- `P[row:row+1]` -> 2-D row block
- `P[:, col]` -> 1-D column selection
- `P[:, col:col+1]` -> 2-D column block
- `P[row, col]` -> scalar element

**Remember:** `P[:][2]` means “take the whole array, then take row 2.” It does not mean “take column 2.”


In [79]:
#NOTE: list slice COPIES data!
l = list(range(10))
s1 = l[2:5]
s1[0] = 100
l, s1

([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [100, 3, 4])

In [101]:
### Tests

P = (100 + np.arange(24, dtype=float)).reshape(6, 4)

#4a.Predict the shape of each: `P[2]`, `P[2:3]`, `P[:, 1]`, `P[:, 1:2]`, `P[2, 1]`, `P[-1, ::2]`, `P[::-1]`.
assert P[2].shape == (4,)
assert P[2:3].shape == (1, 4)
assert P[:, 1].shape == (6,)
assert P[:, 1:2].shape == (6, 1)
assert P[2, 1].shape == ()
assert P[-1, ::2].shape == (2,) # Why??? Why is it not an 2-D array?
assert P[::-1].shape == (6, 4)

# **4b.** `last = P[-1]; last[:] = 0`. Does `P` change? What single edit to the first line prevents it?
last = P[-1]
last[:] = 0
assert np.array_equal(P[-1],last)
last2 = P[-1].copy()
last2[:] = 100
assert np.array_equal(P[-1], last)

# **4c.** `window = P[1:4]; P[2] = -1`. What does `window` show now? 
# What does that tell you about the *direction* of a view relationship?
window = P[1:4]
P[2] = -1
assert np.array_equal(window[1], np.array([-1, -1, -1, -1])) # a slice of array is a view and mutation is bi-directional!


# **4d.** Set the last two days of the last asset to NaN with one assignment. Then predict what happens if `P` had integer dtype.
P[-2:, -1] = np.nan
# if P had integer dtype then it would throw an error as nan and int are incompatible

# **4e.** Predict `P[:][2]` and explain why it isn't column 2.
# it's the 2nd row of P; column 2 is P[:, 2]
assert np.array_equal(P[:][2], np.array([-1, -1, -1, -1]))

### Boolean indexing

**Mental model.** 
1. A comparison yields a same-shape boolean array. 
1. A boolean array of length `n` used along axis 0 selects **rows**. This always returns a 2-D array even with just one row returned.
    - The boolean array must be of the same length as the array axis it's indexing.
1. A mask with the **same shape** as the array selects elements (e.g. `x[x>0]`) and returns a **flattened 1-D** result because selecting individual elements cannot guarantee it will form a rectangular shape. 
1. Combine masks with `&`, `|`, `~` — never `and`/`or`/`not` — and parenthesize, because `&`/`|` bind tighter than `==`/`<`. 
1. Boolean indexing always **copies** by calling `__getitem__`; yet `arr[mask] = value` still modifies `arr`, because that's a `__setitem__` on the original rather than a mutation of a copy.



In [113]:
#**Tests** — use:

tickers = np.array(["AAPL", "MSFT", "AAPL", "GOOG", "MSFT"])
ret = rng.normal(0, 0.02, size=(5, 3))   # 5 tickers × 3 days
# 5a. Predict the shapes of `ret[tickers == "AAPL"]` and `ret[(tickers == "AAPL") | (tickers == "GOOG"), 1:]`.
assert ret[tickers == "AAPL"].shape == (2, 3) # it selects 2 rows and all columns
assert ret[(tickers == "AAPL") | (tickers == "GOOG"), 1:].shape == (3, 2) # it selects 3 rows and 2 cols.

# 5b. Both lines fail. Give the exact reason for each:

# fails because it needs () between two OR conditions as | is higher than comparison so it binds before comparison.
# ret[tickers == "AAPL" | tickers == "GOOG"] 
# correct one should be
ret[(tickers == "AAPL") | (tickers == "GOOG")]
# fails because it needs to use & not "and"; and will lead to one truth value which raises an “ambiguous truth value” error.
#ret[(tickers == "AAPL") and (tickers == "GOOG")]
# correct one should be:
ret[(tickers == "AAPL") & (tickers == "GOOG")] # though this won't have data meeting this condition!

# 5c. Predict the shape of `ret[ret > 0]`. Why isn't it 2-D?
# its shape is (k,) e.g. 1-D array; because the mask is the same shape as the array and it returns a flattened 1-D array.

# 5d. `neg = ret[ret < 0]; neg[:] = 0` — does `ret` change? Now `ret[ret < 0] = 0` — does `ret` change? Explain via `__getitem__` vs `__setitem__`.
# 1) ret doesn't change because boolean indexing copies data! ret[mask] calls __getitem__ and returns a copy!
# 2) it does change ret as it calls __setitem__ on ret directly not on a copy of ret.
# 5e. Winsorize in place: cap returns below −3% at −3% and above +3% at +3% using boolean indexing. Then find the single-function alternative.
ret[ret < -.03] = -.03
ret[ret >.03] = .03
ret = np.clip(ret, -.03, .03)

### Fancy indexing

**Mental model.** 
1. Indexing with integer arrays/lists. 
1. *One* index array selects whole **rows** in the given order (duplicates allowed). 
1. *Two* index arrays are paired elementwise and pick individual elements → 1-D result; for a rectangular block use `arr[rows][:, cols]` or `arr[np.ix_(rows, cols)]`. 
    - Two index arrays must broadcast to a common (not necessarily exactly the same) shape!
1. **Fancy indexing always copies**. 
1. `np.ix_` constructs indices for selecting all combinations of specified rows and columns.
1. Assignment through fancy indexing modifies the original. This plus `argsort` is the mechanism behind "sort/rank one array by another".

In [ ]:
#**Tests** — use 
M = np.arange(1, 21).reshape(4, 5)

#6a. Predict each:
  
assert np.array_equal(M[[3, 1]], np.array([
    [16, 17, 18, 19, 20],
    [6, 7, 8, 9, 10]
    ]))

assert np.array_equal(M[[3, 1], [0, 4]], np.array([16, 10]))

#NOTE: this first selects row 3 and row 1 then from that result it selects col 0 and col 4.
assert np.array_equal(M[[3, 1]][:, [0, 4]], np.array([[16, 20],[6, 10]]))
assert np.array_equal(M[np.ix_([3, 1], [0, 4])], np.array([[16, 20],[6, 10]]))

#- **6b.** Why does `M[[0, 1, 2], [0, 1]]` raise? What rule must the index arrays satisfy?
# ANS: Two index arrays are paired so they must have the same shape to pick individual elements.

# 6c. `sel = M[[0, 1]]; sel[0, 0] = -99` → does `M` change? `M[[0, 1]] = 0` → does `M` change?
#ANS: fancy indexing copies data so first operation doesn't change M. Fancy indexing assignemtn does change M.

#- **6d.** With `P` (6 days × 4 assets), reorder the columns so assets are sorted by last-day price, ascending, in one expression.
P = np.array([
    [10, 11, 12, 13],
    [20, 21, 22, 23],
    [30, 31, 32, 33],
    [40, 41, 42, 43],
    [50, 51, 52, 53],
    [24, 21, 23, 22],
])
order = P[-1].argsort() # use the last row to create a sorting order.
print(P[:, order])

#- 6e. (Stretch) 
v = np.zeros(3)
# each increment is overwritten by the next one so it ends up with just one increment in the end.
v[[0, 0, 0]] += 1 # the RHS is computed once and assigned once
assert np.array_equal(v, np.array([1, 0, 0]))
# Predict `v`. Look up `np.add.at` and explain what it fixes.

# np.add.at performs an unbuffered operation: 
# it applies the ufunc sequentially, one index at a time, directly against v, rather than gather-once/scatter-once:
np.add.at(v, [0, 0, 0], 1) # this increments the 0th element of v 3 times
v
 
 

[[11 13 12 10]
 [21 23 22 20]
 [31 33 32 30]
 [41 43 42 40]
 [51 53 52 50]
 [21 22 23 24]]


array([4., 0., 0.])

### Transposing arrays and swapping axes

**Mental model.** 
1. `.T`, `.transpose(axes)`, `.swapaxes(i, j)` return **views** — no data moves; only shape/strides metadata is rewritten. 
    - For a two-dimensional array, `.T` exchanges rows and columns.
    - For higher-dimensional arrays, `.T` **reverses** the axis order.
1. `transpose(...)` specifies an axis permutation.
1. `swapaxes(...)` exchanges **two** axes.
1. Transposing a one-dimensional array leaves its shape unchanged. `.T` on a 1-D array is a no-op — one axis, nothing to reverse.
1. Reshaping may return a view or require a copy; do not assume either universally. ([numpy.org](https://numpy.org/doc/stable/reference/generated/numpy.transpose.html))
1. `.transpose((2, 0, 1))` reads: "new axis 0 is old axis 2, new axis 1 is old axis 0, new axis 2 is old axis 1". 




In [141]:
a = np.arange(24).reshape(2, 3, 4)
print(a)
b = a.transpose((2, 0, 1))
b

[[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]


array([[[ 0,  4,  8],
        [12, 16, 20]],

       [[ 1,  5,  9],
        [13, 17, 21]],

       [[ 2,  6, 10],
        [14, 18, 22]],

       [[ 3,  7, 11],
        [15, 19, 23]]])

In [142]:
a = np.arange(15).reshape(3, 5)
a @ a.T

array([[ 30,  80, 130],
       [ 80, 255, 430],
       [130, 430, 730]])

In [ ]:
#**Tests**

#7a.`R` is 250 days × 5 assets. # Predict the shapes of `R.T @ R` and `R @ R.T`. 
#ANS: R.T @ R has a shape of (5, 5), R @ R.T has a shape of (250, 250)
# If `R` were demeaned, what is the first one (in finance terms)?
# the first is the unnormalized covariance matrix!

#7b.
A = np.arange(24).reshape(2, 3, 4)

# Predict `A.transpose((2, 0, 1)).shape`, `A.swapaxes(0, 2).shape`, and the *value* of `A.transpose((2, 0, 1))[1, 0, 2]`.
#ANS with assertion: 
assert A.transpose((2, 0, 1)).shape == (4, 2, 3)
assert A.swapaxes(0, 2).shape == (4, 3, 2)
# new[j, k, i] = A[k,i,j] where j=1, k=0, i=2 so A[k,i,j] = A[0, 2, 1]
assert A.transpose((2, 0, 1))[1, 0, 2] == A[0, 2, 1] == 9 



#7c.`T = P.T; T[0, 0] = 999`. Does `P[0, 0]` change?
#ANS: P[0, 0] does change because transpose returns a view.

#7d.
x = np.arange(4)
# Predict `x.T.shape`. 
# ANS with assertion
assert x.T.shape == (4,)
# Give two ways to turn `x` into a `(4, 1)` column vector.
x.reshape(4, 1) # way 1
x.reshape(-1, 1) # way 2
x[:, None] # way 3
x[:, np.newaxis] # way 4


array([[0],
       [1],
       [2],
       [3]])

### Pseudorandom number generation

**Mental model.** 
1. `rng = np.random.default_rng(seed)` returns a `Generator` with its own isolated state (unlike the legacy global `np.random.seed`). 
    - same-seeded generators replay the same stream; a single generator advances.
1. Methods: `standard_normal`, `normal(loc, scale, size)`, `uniform`, `random`, `integers(low, high, size)` (high *exclusive*), `choice`, `permutation` (returns new), `shuffle` (in place), `binomial`, `beta`, `gamma`, `chisquare`. 
1. "Pseudo" = fully deterministic given the seed.



In [ ]:
# **Tests**
#8a.
rng.integers(0, 2, size=10) 
# — which values can appear? 
# ANS: 0 or 1

# Write two different one-liners that produce ±1 steps directly.
rng.choice([-1, 1], size=10)
2*rng.integers(0, 2, size=10) - 1


#8b.Predict both results:  
r1, r2 = np.random.default_rng(7), np.random.default_rng(7)

# same-seeded generators replay the same stream; a single generator advances.
assert (r1.standard_normal(3) == r2.standard_normal(3)).all() == True
assert (r1.standard_normal(3) == r1.standard_normal(3)).all() == False

# 8c Simulate 252 days of returns for 4 assets with mean 0.0004 and std 0.01. 
rets = rng.normal(.0004, .01, (252, 4))
# Predict the shape; compute mean and std per asset. Which axis, and why won't the sample means equal 0.0004?
assert rets.shape == (252, 4)
print(rets.mean(axis=0), rets.std(axis=0))

# 8d. Write a shuffle that leaves `r` intact, and one that destroys its order. 
# Then use `rng.permutation(len(r))` with fancy indexing to shuffle the *rows* of a 2-D array while keeping each row intact.
r = rng.normal(size=100)
# 8e. Two reasons to prefer `default_rng` over `np.random.seed(0); np.random.randn(...)` in strategy research.

 # 8c. rng.normal(0.0004, 0.01, size=(252, 4)) → (252, 4); .mean(axis=0); standard error ≈ 0.01/√252 ≈ 0.0006, larger than the mean itself. 8d. rng.permutation(r); rng.shuffle(r); M[rng.permutation(len(M))]. 8e. Isolated state (no hidden coupling between functions/notebooks); better and faster algorithm; explicit seeds make backtests reproducible.

[ 0.00059236 -0.00022199  0.00048872 -0.00089655] [0.01084484 0.01000188 0.00979553 0.00986618]


### Other Topics
1. The output shape is the input shape with every reduced axis removed whenever `np.func(axis=n)` is passed.



---

## 4.3 Universal functions

**Mental model.** A ufunc is a compiled elementwise function with broadcasting built in. Operators are ufuncs in disguise (`+` → `np.add`, `>` → `np.greater`). Unary: `sqrt, exp, log, log1p, expm1, abs, sign, floor, ceil, rint, isnan, isfinite, isinf`. Binary: `add, maximum/fmax, minimum/fmin, mod, power, copysign`, comparisons. Some return several arrays (`modf`). All accept `out=` for in-place writes, subject to dtype casting rules. Invalid math yields `nan`/`inf` plus a `RuntimeWarning` — not an exception.

**Tests**
- **9a.** `r = np.array([0.01, -0.02, 0.0, 1e-10])`. Compute log returns as `np.log(1 + r)` and as `np.log1p(r)`. Which is more accurate for the last element, and why? What's the inverse of `log1p`?
- **9b.** Predict `np.maximum([1, np.nan, 3], [2, 2, np.nan])` and `np.fmax(...)` on the same inputs. Which do you want for an elementwise max of two signal series with gaps?
- **9c.** Predict `np.rint([0.5, 1.5, 2.5, -0.5])`. Name the rounding rule; how does it differ from `np.floor(x + 0.5)`?
- **9d.** Predict both outputs of `np.modf(np.array([-2.75, 3.5]))`.
- **9e.** `x = np.arange(5, dtype=float); np.sqrt(x, out=x)` — predict `x`. Repeat with `x = np.arange(5)` (int). What happens, and which rule enforces it?
- **9f.** Turn momentum `m = np.array([-0.08, 0.0, 0.03])` into a −1/0/+1 signal with one ufunc.

---

## 4.4 Array-oriented programming

### Grid evaluation with `np.meshgrid` (section intro)

**Mental model.** `X, Y = np.meshgrid(x, y)` gives two `(len(y), len(x))` arrays — `X` repeats `x` along rows, `Y` repeats `y` down columns — so `f(X, Y)` evaluates `f` at every (x, y) pair. Broadcasting does the same thing without materializing the grids: `f(x, y[:, np.newaxis])`.

**Tests**
- **10a.** `x = np.array([1, 2, 3]); y = np.array([10, 20]); X, Y = np.meshgrid(x, y)`. Predict `X`, `Y`, `X + Y`. Then reproduce `X + Y` with broadcasting only.
- **10b.** `spots = np.array([90, 100, 110]); strikes = np.array([95, 105])`. Produce the 2×3 grid of call payoffs `max(S − K, 0)` with meshgrid plus one ufunc. Predict the values.

### Conditional logic: `np.where`

**Mental model.** `np.where(cond, x, y)` is a vectorized ternary; `x`/`y` may be scalars or arrays and are broadcast. Nest for several conditions (`np.select` is cleaner for many). Both branches are fully evaluated before selection — no short-circuiting. With only `cond`, `where` returns a *tuple* of index arrays, not a mask.

**Tests**
- **11a.** `mom = np.array([0.12, -0.03, 0.0, 0.07])`. Predict `np.where(mom > 0.05, 1, np.where(mom < -0.02, -1, 0))`.
- **11b.** `R` is `(60, 8)` with NaNs. Fill each NaN with its *column* mean in one `np.where`. Which broadcasting fact makes it work? Why would *row* means need an extra step?
- **11c.** Predict `np.where(np.array([True, False, True]))`. What type is it?
- **11d.** `r = np.array([-0.01, 0.02, 0.0])`. Why does `np.where(r > 0, np.log(r), 0.0)` warn even though the result is correct? Write a version that doesn't.

### Mathematical and statistical methods

**Mental model.** `sum, mean, std, var, min, max, argmin, argmax, cumsum, cumprod` as methods or functions. `axis=k` means "collapse axis k": on a (days, assets) matrix, `axis=0` gives one number per asset, `axis=1` one per day. Slogan: *the axis you name is the one that disappears.* `cumsum`/`cumprod` don't reduce — same shape. `argmax` with no axis returns a *flat* index. `np.std` defaults to `ddof=0`; pandas defaults to `ddof=1` — the classic "why don't my numbers match". NaN poisons plain reductions; use `nanmean, nansum, nanstd, nanmax` — and note `nansum` of all-NaN is `0.0`, not NaN.

**Tests** — use `R = rng.normal(0.0005, 0.01, size=(250, 3))`.
- **12a.** Write "average return per asset" and "equal-weight portfolio return per day". Predict both shapes.
- **12b.** `R.std()` and `pd.DataFrame(R).std()` disagree. Why, and how do you make NumPy match?
- **12c.** Compute cumulative wealth `W` from 1.0 for each asset. Predict `W.shape`; which row is total growth? Then find the *day index* of each asset's peak.
- **12d.** Predict `np.arange(6).reshape(2, 3).cumsum(axis=1)`.
- **12e.** `R.argmax()` returns one integer. Convert it to `(day, asset)`.
- **12f.** Predict `np.array([1, np.nan, 3]).mean()`, `np.nanmean(...)`, and `np.nansum(np.array([np.nan, np.nan]))`.
- **12g.** (Stretch) Drawdown per asset: `W / np.maximum.accumulate(W, axis=0) - 1`. Explain what `np.maximum.accumulate` does and why `axis=0`. Then extract max drawdown per asset.

### Methods for boolean arrays

**Mental model.** `True` is 1, `False` is 0 under arithmetic — so `.sum()` counts and `.mean()` gives a proportion. `.any()`/`.all()` take `axis`; on numeric arrays, nonzero counts as `True`.

**Tests**
- **13a.** Hit rate (fraction of up days) per asset, one expression.
- **13b.** `R` has NaNs. Return only the days with no missing values, one expression.
- **13c.** Predict `np.array([0.0, 0.1, -2]).all()` and `.any()`.
- **13d.** How many days did *all three* assets rise?

### Sorting

**Mental model.** `arr.sort()` sorts in place and returns `None`; `np.sort(arr)` returns a sorted copy. Default axis is the *last*. Sorting a 2-D array along `axis=0` sorts each column independently — rows no longer belong to the same observation. `argsort` returns the sorting permutation; use it with fancy indexing to sort one array by another. `argsort` applied twice yields ranks. Descending: `np.sort(a)[::-1]` or `np.argsort(-a)`. **NaNs sort to the end.**

**Tests**
- **14a.** `x = np.array([3, 1, 2]); y = x.sort()`. Predict `y` and `x`.
- **14b.** `M = np.array([[3, 1], [1, 5], [2, 0]])`. Predict `np.sort(M, axis=0)`, `np.sort(M, axis=1)`, `np.sort(M)`.
- **14c.** `mom = np.array([0.05, 0.12, -0.02, 0.08]); tk = np.array(["A", "B", "C", "D"])`. Return the two tickers with highest momentum, highest first.
- **14d.** Predict `mom.argsort().argsort()` and explain in one sentence why double argsort gives ranks.
- **14e.** Estimate the 5th-percentile daily return of `r = rng.normal(0, 0.01, 1000)` by sorting and indexing. Compare with `np.quantile(r, 0.05)`; explain a small difference.
- **14f.** `z = np.array([0.5, np.nan, -1.0, 2.0])`. Predict `np.argsort(z)[::-1][:2]`. Why is this a bug in a "long the top-2" strategy, and how do you fix it?

### Unique and other set logic

**Mental model.** `np.unique` returns *sorted* uniques; `return_counts`, `return_index`, `return_inverse` add extras. `inverse` is a vector of integer codes such that `u[inv]` reconstructs the input — this is exactly how pandas Categoricals and groupby work internally. Membership: `np.isin(arr, values)`. Set ops on 1-D: `intersect1d, union1d, setdiff1d, setxor1d`.

*NumPy 2 note:* the book's `np.in1d` is deprecated — use `np.isin`.

**Tests**
- **15a.** `tk = np.array(["MSFT", "AAPL", "MSFT", "GOOG", "AAPL", "MSFT"])`. Predict `np.unique(tk, return_counts=True)`.
- **15b.** Select rows of a returns matrix whose ticker is in `["AAPL", "GOOG"]` using one membership function.
- **15c.** Rebalance: `held = np.array(["AAPL", "MSFT", "TSLA"]); target = np.array(["MSFT", "NVDA", "AAPL"])`. Compute sells, buys, holds with three set functions; predict each.
- **15d.** `u, inv = np.unique(tk, return_inverse=True)`. Predict `inv`; verify `u[inv]` equals `tk`. Then count per ticker using `inv` and `np.bincount`.

---

## 4.5 File input and output with arrays

**Mental model.** `np.save(path, arr)` writes `.npy` (extension appended if missing) preserving dtype and shape exactly; `np.load` reads it. `np.savez(path, a=..., b=...)` bundles arrays into `.npz`; loading returns a dict-like object that reads each array lazily on key access; `savez_compressed` compresses. Text (`savetxt`/`loadtxt`) loses dtype fidelity — the book leaves CSV to pandas.

**Tests**
- **16a.** `np.save("prices", P)`. What filename appears? Predict what `np.load("prices")` does.
- **16b.** Round-trip an `int32` array through `save/load` and through `savetxt/loadtxt`. Predict the dtype after each.
- **16c.** `arch = np.load("bundle.npz")`. How do you list the contents, and when is each array actually read?
- **16d.** Three reasons `.npy` beats CSV as a cache for a 2,500 × 3,000 returns matrix. Look up `mmap_mode` for the third.

---

## 4.6 Linear algebra

**Mental model.** `*` is elementwise; matrix multiply is `@` (or `np.dot`). `(n, k) @ (k,)` → `(n,)`; `(k,) @ (k,)` → scalar. `np.linalg` provides `inv, solve, lstsq, det, eig, svd, qr, pinv`. Prefer `solve(A, b)` over `inv(A) @ b`.

**Tests**
- **17a.** `w = np.array([0.5, 0.3, 0.2])`, `R` is `(250, 3)`. Predict the shape of `R @ w`. Why doesn't `R * w` raise, what shape is it, and how do you recover `R @ w` from it?
- **17b.** `C = np.cov(R, rowvar=False)`. Predict `C.shape`, then the shape after each step of `w @ C @ w`. What is the result financially?
- **17c.** Predict `np.array([1, 2, 3]) @ np.array([4, 5, 6])` and `np.array([[1, 2, 3]]) @ np.array([[4, 5, 6]])`.
- **17d.** OLS beta: `X` is `(250, 2)` (column of ones, market returns), `y` the asset returns. Compute coefficients via `np.linalg.solve` on the normal equations and via `np.linalg.lstsq`; confirm they match. Why avoid `inv(X.T @ X)`?
- **17e.** What does `np.linalg.inv(np.array([[1, 2], [2, 4]]))` do, and why?

---

## 4.7 Random walks → a Monte Carlo capstone

**Mental model.** The book's pattern: `draws → np.where → cumsum`, then vectorize across many walks by adding a leading axis and using `axis=1`. First-crossing time is `(cond).argmax(axis=1)` — valid because `argmax` returns the *first* `True` — but only after filtering with `.any(axis=1)`, because `argmax` on an all-`False` row silently returns `0`.

**Tests (in order — they build)**
- **18a.** Simulate 10,000 paths × 252 days of returns `~ N(0.0003, 0.012)`; convert to price paths from 100 with `cumprod`. Predict the shape and `nbytes` in MB *before* running.
- **18b.** Fraction of paths ending above 120 — one expression.
- **18c.** For each path that ever closes below 90, the first day it happened; and how many paths never did. Why is `(paths < 90).argmax(axis=1)` alone wrong?
- **18d.** Max drawdown per path via `np.maximum.accumulate(..., axis=1)`, then the 5th percentile of those across paths.
- **18e.** Replace the normal draws with ±1% coin flips via `rng.choice`; confirm the book's `integers + np.where` idiom gives the same distribution.
- **18f.** Conceptual: why is the 2-D formulation faster than a Python loop over paths, and when would memory force you to chunk?

---

## Integrated capstone — one cross-sectional factor step (what QuantConnect work actually looks like)

```python
rng = np.random.default_rng(0)
R = rng.normal(0, 0.02, size=(60, 8))        # 60 days × 8 assets
R[rng.random(R.shape) < 0.05] = np.nan       # ~5% missing
```
- **C1.** Trailing 20-day momentum as of day 40 (rows 20–39): compound `1 + R` down the time axis, treating NaN as "no return". Predict the shape.
- **C2.** Cross-sectional z-score of that vector, NaN-aware.
- **C3.** Long top 2 / short bottom 2 by z-score, *excluding* NaN assets correctly (recall 14f).
- **C4.** Build `w` of shape `(8,)` with +0.5/−0.5 in the chosen slots via fancy-index assignment; compute the day-40 portfolio return ignoring NaN.
- **C5.** Repeat C1–C4 for every day 20..59 (a loop over days is fine; no loop over assets). Which steps needed `axis=0`, which `axis=1`, and where could a view-vs-copy issue bite you?

Finish C5 and you own the NumPy layer of a factor backtest; what remains is pandas alignment and QC plumbing.




Answer key — attempt everything first
0a. Homogeneous dtype → no per-element type dispatch or boxing; the loop runs in compiled C over contiguous, cache-friendly memory. 0b. 8,000,000 bytes; 4,000,000 with int32.

1a. float64; a Unicode dtype (<U21); int64. 1b. (2, 1, 3), ndim 3. 1c. TypeError — second positional arg is dtype; use np.zeros((3, 4)). 1d. a[0] is 99 (asarray returned a itself); c[0] is 0. 1e. np.full((5, 3), 100.0); np.full_like(prices, np.nan); the int template keeps int dtype, so NaN becomes a garbage integer plus a RuntimeWarning.

2a. [-1, 1, 2, 0] — truncation toward zero. 2b. [3.5, 4.0, 1000.0, -0.25]. 2c. [251, 252, 253] (uint8 wraps). x - 300: NumPy 2 raises OverflowError; NumPy 1.x upcasts to uint16 and wraps. 2d. True division always produces float; floor division keeps integer dtype. 2e. 60 MB vs 30 MB; float32 ≈ 7 significant digits — fine for storing returns, risky when accumulating sums/products over thousands of terms. 2f. 0 — astype always copies.

3a. P[1:] / P[:-1] - 1, shape (3, 3). 3b. Works: demeans each asset over time. Fails: (5,3) vs (5,) can't align from the right. Works: demeans each day cross-sectionally. 3c. Line 2 now runs and subtracts day means along the asset axis — silent garbage. Square matrices hide broadcasting bugs; always test with non-square shapes. 3d. (2, 3): [[10, 20, 30], [20, 40, 60]]. 3e. False; use np.isclose/np.allclose.

4a. (4,), (1, 4), (6,), (6, 1), scalar with shape (), (2,), (6, 4). 4b. Yes; last = P[-1].copy(). 4c. window[1] is now all −1 — views share memory, so writes are visible in both directions. 4d. P[-2:, -1] = np.nan; with int dtype: ValueError (can't convert NaN to integer). 4e. Row 2 — P[:] is a view of the whole array, then [2] indexes axis 0 again.

5a. (2, 3); (3, 2). 5b. | binds tighter than ==, so NumPy first attempts "AAPL" | tickers → TypeError; parenthesize. and requests the truth value of a 5-element array → ValueError "ambiguous"; use &. 5c. 1-D (k,) — a same-shape mask picks elements, flattened row-major. 5d. No (ret[mask] returned a copy); yes (ret[mask] = 0 is __setitem__ on the original). 5e. ret[ret < -0.03] = -0.03; ret[ret > 0.03] = 0.03; or np.clip(ret, -0.03, 0.03).

6a. [[16..20], [6..10]]; [16, 10] (elements (3,0) and (1,4)); [[16, 20], [6, 10]]; same. 6b. IndexError — index arrays must broadcast to a common shape; (3,) and (2,) don't. 6c. No (copy); yes (assignment hits the original). 6d. P[:, P[-1].argsort()]. 6e. [1, 0, 0] — the RHS is computed once and assigned once; np.add.at(v, [0, 0, 0], 1) gives 3.

7a. (5, 5) and (250, 250); the first is the unnormalized covariance matrix. 7b. (4, 2, 3); (4, 3, 2); 9 — new[i, j, k] = A[j, k, i], so A[0, 2, 1] = 9. 7c. Yes — a view. 7d. (4,), unchanged; x[:, np.newaxis] or x.reshape(-1, 1).

8a. Only 0 and 1; rng.choice([-1, 1], size=10) or 2 * rng.integers(0, 2, size=10) - 1. 8b. True; False — same-seeded generators replay the same stream; a single generator advances. 8c. rng.normal(0.0004, 0.01, size=(252, 4)) → (252, 4); .mean(axis=0); standard error ≈ 0.01/√252 ≈ 0.0006, larger than the mean itself. 8d. rng.permutation(r); rng.shuffle(r); M[rng.permutation(len(M))]. 8e. Isolated state (no hidden coupling between functions/notebooks); better and faster algorithm; explicit seeds make backtests reproducible.

9a. log1p — 1 + 1e-10 already loses digits before the log. Inverse: np.expm1. 9b. [2, nan, nan]; [2, 2, 3]; fmax to ignore gaps. 9c. [0., 2., 2., -0.] — round-half-to-even; floor(x + 0.5) always rounds .5 up. 9d. [-0.75, 0.5] and [-2., 3.] — the fractional part carries the sign. 9e. [0, 1, √2, √3, 2]; the int version raises UFuncTypeError — float64 output can't be cast to int64 under the same_kind rule. 9f. np.sign(m) → [-1, 0, 1].

10a. X = [[1,2,3],[1,2,3]], Y = [[10,10,10],[20,20,20]], X + Y = [[11,12,13],[21,22,23]]; x + y[:, np.newaxis]. 10b. np.maximum(S - K, 0) → [[0, 5, 15], [0, 0, 5]].

11a. [1, -1, 0, 1]. 11b. np.where(np.isnan(R), np.nanmean(R, axis=0), R); the (8,) vector broadcasts against (60, 8). Row means are (60,) and need [:, np.newaxis]. 11c. (array([0, 2]),) — a tuple. 11d. Both branches are evaluated over all elements, so np.log sees 0 and negatives. Fix: out = np.zeros_like(r); m = r > 0; out[m] = np.log(r[m]), or np.log(r, where=r > 0, out=np.zeros_like(r)).

12a. R.mean(axis=0) → (3,); R.mean(axis=1) → (250,). 12b. ddof 0 vs 1; R.std(ddof=1). 12c. W = np.cumprod(1 + R, axis=0), (250, 3); W[-1]; W.argmax(axis=0). 12d. [[0, 1, 3], [3, 7, 12]]. 12e. np.unravel_index(R.argmax(), R.shape). 12f. nan; 2.0; 0.0 — a silent error source. 12g. Running maximum down time; drawdown is measured against the peak so far, hence axis 0; dd.min(axis=0).

13a. (R > 0).mean(axis=0). 13b. R[~np.isnan(R).any(axis=1)]. 13c. False (0.0 is falsy); True. 13d. (R > 0).all(axis=1).sum().

14a. y is None; x is [1, 2, 3]. 14b. [[1, 0], [2, 1], [3, 5]] (columns sorted independently — rows scrambled); [[1, 3], [1, 5], [0, 2]]; same as axis=1. 14c. tk[np.argsort(mom)[::-1][:2]] → ["B", "D"]. 14d. [1, 3, 0, 2]; the first argsort says "which element goes in position i", the second inverts that permutation to say "which position element i lands in" — its rank. 14e. np.sort(r)[int(0.05 * len(r))]; np.quantile interpolates between order statistics by default. 14f. [1, 3] — NaN sorts last, so reversing puts the NaN asset in the top slot. Fix: valid = ~np.isnan(z); idx = np.flatnonzero(valid)[np.argsort(z[valid])], then take from the end of idx.

15a. (['AAPL', 'GOOG', 'MSFT'], [2, 1, 3]) — sorted. 15b. R[np.isin(tk, ["AAPL", "GOOG"])]. 15c. np.setdiff1d(held, target) → ['TSLA']; np.setdiff1d(target, held) → ['NVDA']; np.intersect1d(held, target) → ['AAPL', 'MSFT']. 15d. inv = [2, 0, 2, 1, 0, 2]; np.bincount(inv) → [2, 1, 3].

16a. prices.npy; np.load("prices") → FileNotFoundError (load does not append the extension). 16b. int32 preserved; float64 after loadtxt. 16c. arch.files; each array is read on arch["key"]. 16d. Exact dtype/shape; no parsing → orders of magnitude faster; np.load(..., mmap_mode="r") reads slices without loading the whole file.

17a. (250,); R * w broadcasts (3,) across columns → (250, 3) of weighted contributions; (R * w).sum(axis=1) equals R @ w. 17b. (3, 3); (3,) @ (3, 3) → (3,), then @ (3,) → scalar; portfolio variance. 17c. 32; ValueError — (1,3) @ (1,3) core dimensions mismatch. 17d. np.linalg.solve(X.T @ X, X.T @ y) and np.linalg.lstsq(X, y, rcond=None)[0]; explicit inversion is slower and amplifies rounding error when X.T @ X is ill-conditioned. 17e. LinAlgError: Singular matrix — rows are linearly dependent.

18a. paths = 100 * np.cumprod(1 + rng.normal(0.0003, 0.012, size=(10_000, 252)), axis=1); (10000, 252); 10000 × 252 × 8 ≈ 20.2 MB. 18b. (paths[:, -1] > 120).mean(). 18c. hit = (paths < 90).any(axis=1); first = (paths[hit] < 90).argmax(axis=1); (~hit).sum(); unfiltered, never-hit rows report day 0. 18d. dd = paths / np.maximum.accumulate(paths, axis=1) - 1; mdd = dd.min(axis=1); np.quantile(mdd, 0.05). 18e. 0.01 * rng.choice([-1, 1], size=(n, 252)) ≡ 0.01 * np.where(rng.integers(0, 2, size=(n, 252)) == 0, -1, 1). 18f. One C loop over a contiguous block vs 10,000 Python iterations with per-call overhead; when nbytes approaches RAM, simulate in row chunks and aggregate.

C1. np.nanprod(1 + R[20:40], axis=0) - 1, shape (8,). C2. (mom - np.nanmean(mom)) / np.nanstd(mom). C3. Mask NaNs first as in 14f; idx[-2:] long, idx[:2] short. C4. w = np.zeros(8); w[long] = 0.5; w[short] = -0.5; np.nansum(R[40] * w). C5. Compounding and z-scores collapse time (axis=0) inside each window; if you stack the daily portfolio returns into a matrix, the asset axis collapses with axis=1. R[t-20:t] is a view — fine to read, but any in-place NaN fill inside the window silently modifies R itself.

In [14]:
128960 * 5.5

709280.0

In [15]:
461.8*0.93-70.928-173.8 - 54

130.74600000000004

In [164]:
2332700.58 - 1350071.42 - 1000000

-17370.83999999985

In [165]:
3179.74 + 280000 - 17370.84

265808.89999999997